## Clip SEVIRI usate nel pretraining VideoMAE

Il notebook legge direttamente dagli Zarr le 100 clip casuali e validate del manifest, senza etichette EMMA/GPM/RDT. La prima riga mostra lo stesso istante nei sette canali; la seconda mostra quattro differenze spettrali diagnostiche. **Play** avanza lungo i 16 frame RSS a cinque minuti.

In [ ]:
from pathlib import Path
from functools import lru_cache
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from emma_gpm.seviri import THERMAL_CHANNELS
from emma_gpm.seviri_clips import SeviriZarrClipReader

ZARR_ROOT = Path(os.environ.get(
    'SEVIRI_ZARR_ROOT',
    r'E:\Datasets\Deep_convection\processed\msg_seviri\rss_7ch_bt_emma_30N70N',
))
MANIFEST = ROOT / 'catalogs' / 'seviri_pretraining_random_100.csv'
manifest = pd.read_csv(MANIFEST)
print(f'Zarr: {ZARR_ROOT}')
print(f'Clip disponibili: {len(manifest)}')
display(manifest)

In [ ]:
reader = SeviriZarrClipReader(ZARR_ROOT)
@lru_cache(maxsize=4)
def load_clip(clip_id):
    row = manifest.loc[manifest.clip_id == clip_id].iloc[0]
    return reader.load(
        row.start_time,
        origin_y=int(row.origin_y),
        origin_x=int(row.origin_x),
    )

channel_labels = [
    'WV 6.2 µm', 'WV 7.3 µm', 'IR 8.7 µm', 'IR 9.7 µm',
    'IR 10.8 µm', 'IR 12.0 µm', 'IR 13.4 µm',
]
BT_MIN_K, BT_MAX_K = 180.0, 330.0
spectral_differences = [
    ('WV 6.2 − IR 10.8', 0, 4, -80.0, 20.0),
    ('IR 8.7 − IR 10.8', 2, 4, -15.0, 15.0),
    ('IR 10.8 − IR 12.0', 4, 5, -10.0, 10.0),
    ('IR 10.8 − IR 13.4', 4, 6, -20.0, 40.0),
]
clip_selector = widgets.Dropdown(
    options=manifest.clip_id.tolist(), description='Clip:', layout=widgets.Layout(width='320px')
)
play = widgets.Play(
    value=0, min=0, max=15, step=1, interval=550, description='Play'
)
frame_slider = widgets.IntSlider(
    value=0, min=0, max=15, step=1, description='Frame:', continuous_update=False,
    layout=widgets.Layout(width='520px')
)
widgets.jslink((play, 'value'), (frame_slider, 'value'))
output = widgets.Output()

def render(*_):
    clip_id = clip_selector.value
    frame = frame_slider.value
    clip = load_clip(clip_id)
    row = manifest.loc[manifest.clip_id == clip_id].iloc[0]
    timestamp = pd.Timestamp(row.start_time) + pd.Timedelta(minutes=5 * frame)
    with output:
        clear_output(wait=True)
        fig = plt.figure(figsize=(23, 8.5), constrained_layout=True)
        outer = fig.add_gridspec(2, 1)
        top = outer[0].subgridspec(1, 7)
        bottom = outer[1].subgridspec(1, 4)
        axes = [fig.add_subplot(top[0, index]) for index in range(7)]
        difference_axes = [fig.add_subplot(bottom[0, index]) for index in range(4)]
        for channel, (axis, label) in enumerate(zip(axes, channel_labels)):
            image = axis.imshow(
                clip[frame, channel], cmap='turbo_r', vmin=BT_MIN_K, vmax=BT_MAX_K
            )
            axis.set_title(label, fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
            fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label='K')
        for axis, (label, minuend, subtrahend, vmin, vmax) in zip(
            difference_axes, spectral_differences
        ):
            difference = clip[frame, minuend] - clip[frame, subtrahend]
            image = axis.imshow(
                difference, cmap='RdBu_r',
                norm=TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax),
            )
            axis.set_title(label, fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
            fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label='ΔK')
        fig.suptitle(f'{clip_id} — {timestamp:%Y-%m-%d %H:%M UTC} — frame {frame + 1}/16', fontsize=15)
        display(fig)
        plt.close(fig)

frame_slider.observe(render, names='value')
clip_selector.observe(lambda change: (setattr(frame_slider, 'value', 0), render()), names='value')
display(widgets.HBox([clip_selector, play, frame_slider]), output)
render()

La prima riga usa la stessa scala assoluta e fissa (180–330 K) per tutti i canali, tutte le clip e tutti i frame. La seconda riga mostra differenze spettrali diagnostiche con intervalli fisici prefissati e centrati sullo zero; queste differenze sono derivate per l'ispezione e non vengono aggiunte all'input VideoMAE.

## Sequenza sull'intero dominio mediterraneo ed europeo

I video seguenti mostrano una sola sequenza RSS completa sul dominio **20°W–40°E, 30–60°N**. I limiti longitudinali coincidono con il dominio EMMA usato nel progetto. Le sette bande e le quattro differenze spettrali sono file separati, ma condividono gli stessi 16 istanti a cadenza di cinque minuti.

In [ ]:
from IPython.display import Video

MEDITERRANEAN_CLIP_ID = 'random_000'
MEDITERRANEAN_VIDEO_DIR = (
    ROOT / 'figures' / 'videomae_pretraining' / f'mediterranean_{MEDITERRANEAN_CLIP_ID}'
)
mediterranean_index = pd.read_csv(MEDITERRANEAN_VIDEO_DIR / 'video_index.csv')
missing_videos = [
    name for name in mediterranean_index.path
    if not (MEDITERRANEAN_VIDEO_DIR / name).exists()
]
if missing_videos:
    raise FileNotFoundError(f'Video mancanti: {missing_videos}')
display(mediterranean_index[['title', 'path', 'vmin_k', 'vmax_k']])

In [ ]:
mediterranean_selector = widgets.Dropdown(
    options=[(row.title, row.path) for row in mediterranean_index.itertuples()],
    description='Video:',
    layout=widgets.Layout(width='520px'),
)
mediterranean_output = widgets.Output()

def show_mediterranean_video(*_):
    with mediterranean_output:
        clear_output(wait=True)
        path = MEDITERRANEAN_VIDEO_DIR / mediterranean_selector.value
        display(Video(filename=str(path), embed=False, width=1050, html_attributes='controls loop'))

mediterranean_selector.observe(show_mediterranean_video, names='value')
display(mediterranean_selector, mediterranean_output)
show_mediterranean_video()

In [ ]:
# Funzione opzionale per produrre gli undici video per un'altra clip del manifest.
import subprocess

def render_mediterranean_videos(clip_id):
    command = [
        sys.executable,
        str(ROOT / 'scripts' / 'render_seviri_mediterranean_videos.py'),
        '--manifest', str(MANIFEST),
        '--clip-id', clip_id,
    ]
    subprocess.run(command, cwd=ROOT, check=True)
    return ROOT / 'figures' / 'videomae_pretraining' / f'mediterranean_{clip_id}'

# Esempio (richiede alcuni minuti):
# render_mediterranean_videos('random_001')

## Campionamento spaziale senza sovrapposizioni

Il nuovo manifest applica IoU zero tra tutte le clip che condividono frame: stessa regione riutilizzabile soltanto dopo la fine della clip precedente. La preparazione usa il training split esistente, senza materializzare pixel e senza avviare nuovi training. Il dominio è 763×1.762 pixel; la ricerca casuale limita le sovrapposizioni ma non garantisce il numero massimo di crop impacchettabili.

La cella seguente è autonoma: mostra il riepilogo, le clip per giornata e le diagnostiche di copertura. Rieseguirla per rileggere i risultati. I grafici di campionamento usano **indici Zarr nativi**, mentre le clip restituite dal loader sono sempre north-up e west-left.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image as NotebookImage

SPACED_ROOT = Path.cwd()
if not (SPACED_ROOT / "src").exists() and (SPACED_ROOT.parent / "src").exists():
    SPACED_ROOT = SPACED_ROOT.parent
SPACED_DIR = SPACED_ROOT / "results/videomae_pretraining_smoke_masking_sweep/spaced_samples"
if not (SPACED_DIR / "dataset_report.json").exists():
    print(f"Preparazione non ancora conclusa. Rieseguire questa cella; log: {SPACED_DIR / 'preparation.log'}")
else:
    spaced_report = json.loads((SPACED_DIR / "dataset_report.json").read_text())
    spaced_manifest = pd.read_csv(SPACED_DIR / "train_spaced_all.csv")
    print(f"Dominio: {spaced_report['grid_height']} × {spaced_report['grid_width']} = "
          f"{spaced_report['pixels_per_frame_per_channel']:,} pixel per frame e canale")
    print(f"Clip accettate: {spaced_report['accepted_samples']}; sottoinsieme piccolo: {spaced_report['small_samples']}; "
          f"rapporto: {spaced_report['growth_ratio']:.2f}×")
    print(f"IoU massima fra clip che condividono frame: {spaced_report['maximum_concurrent_iou']:.6f}; "
          f"coppie controllate: {spaced_report['concurrent_pairs_checked']}")
    print(f"Parametri: {spaced_report['sampling']}")
    spaced_days = spaced_manifest.assign(giorno=pd.to_datetime(spaced_manifest.start_time, utc=True).dt.strftime("%Y-%m-%d"))
    display(spaced_days.groupby("giorno").agg(clip=("clip_id", "size"), partenze=("start_time", "nunique")))
    display(spaced_manifest.head(10))
    display(NotebookImage(filename=str(SPACED_DIR / "sampling_diagnostics.png")))
    if (SPACED_DIR / "overlap_comparison.json").exists():
        overlap_report = json.loads((SPACED_DIR / "overlap_comparison.json").read_text())
        print(f"Riutilizzo di osservazioni pixel/frame: "
              f"casuali {overlap_report['reference']['repeated_pixel_frame_fraction']:.1%}, "
              f"separati {overlap_report['spaced']['repeated_pixel_frame_fraction']:.1%}.")
        display(NotebookImage(filename=str(SPACED_DIR / "overlap_comparison.png")))


## Monitoraggio VideoMAE V2 Small e loss per epoca

Celle autonome: **Aggiorna** oppure **Auto (30 s)**. Coda: `results/videomae_v2_small_scaling`.

I primi tre run usano 2.000 clip e masking encoder 50/75/90%; i successivi 4.000 e 8.000 clip usano il masking selezionato sulla validation comune. Tutti partono da zero per 150 epoche, con decoder masking **50% running cell**. La tabella contiene soltanto Stato ed Epoche. Training continuo, validation tratteggiata; default **Proprio masking** dell'encoder, con validation registrata ogni epoca. Il confronto **90% per tutti** è disponibile lungo tutto il training; altre percentuali comuni richiedono valutazioni aggiuntive e possono essere assenti.

Le loss sono in **K²**, usando le statistiche del rispettivo training manifest: la scala rimane confrontabile anche quando cambia la dimensione del dataset. I compiti con encoder masking diversi restano di difficoltà diversa. Dopo la selezione compare anche la curva di validation in funzione del numero di clip. Con 150 epoche, i dataset grandi richiedono più aggiornamenti: il confronto non è a compute uguale.

Estensione del 4 ottobre: i run originali da 4.000/8.000 al 75% sono conservati. Due nuove prove **large4000_mask90 / large8000_mask90**, da zero per 150 epoche, usano encoder90% e decoder50%, con gli stessi manifest, statistiche, seed e holdout. Il monitor segue automaticamente la nuova coda; la curva per dimensione confronta encoder75% e encoder90%.

In [ ]:
from pathlib import Path
import asyncio
import json
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Se questa cella viene rieseguita, ferma l'eventuale aggiornamento precedente.
_previous_monitor_task = globals().get("training_monitor_task")
if _previous_monitor_task is not None and not _previous_monitor_task.done():
    _previous_monitor_task.cancel()
training_monitor_task = None

MONITOR_ROOT = Path.cwd()
if not (MONITOR_ROOT / "src").exists() and (MONITOR_ROOT.parent / "src").exists():
    MONITOR_ROOT = MONITOR_ROOT.parent
MONITOR_QUEUE = MONITOR_ROOT / "results/videomae_v2_small_scaling"
MONITOR_RUNS = [
    ("mask50", "V2 · 2.000 clip · encoder 50%", "tab:blue"),
    ("mask75", "V2 · 2.000 clip · encoder 75%", "tab:orange"),
    ("mask90", "V2 · 2.000 clip · encoder 90%", "tab:green"),
    ("large4000", "V2 · 4.000 clip · encoder 75%", "tab:red"),
    ("large8000", "V2 · 8.000 clip · encoder 75%", "tab:purple"),
    ("large4000_mask90", "V2 · 4.000 clip · encoder 90%", "tab:cyan"),
    ("large8000_mask90", "V2 · 8.000 clip · encoder 90%", "tab:brown"),
]

def monitor_read_json(path):
    try:
        return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        # Un file può non esistere ancora o essere aggiornato proprio ora.
        return {}

def monitor_read_epochs(path):
    try:
        frame = pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError):
        return pd.DataFrame()
    if not {"epoch", "mean_loss"}.issubset(frame.columns):
        return pd.DataFrame()
    for column in frame.columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return (frame.dropna(subset=["epoch", "mean_loss"])
                 .drop_duplicates("epoch", keep="last")
                 .sort_values("epoch"))

def monitor_log_tail(path):
    try:
        with path.open("rb") as handle:
            handle.seek(0, 2)
            handle.seek(max(0, handle.tell() - 4096))
            lines = handle.read().decode("utf-8", errors="replace").splitlines()
        return lines[-1][:200] if lines else "Avvio del processo…"
    except FileNotFoundError:
        return "Log non ancora disponibile."

print(f"Directory monitorata: {MONITOR_QUEUE}")


In [ ]:
monitor_refresh_button = widgets.Button(description="Aggiorna", icon="refresh")
monitor_auto = widgets.Checkbox(value=False, description="Auto (30 s)")
monitor_validation_mode = widgets.Dropdown(
    options=[("Proprio masking", "own"), ("50% per tutti", .5), ("75% per tutti", .75), ("90% per tutti", .9)],
    value="own", description="Validation:", layout=widgets.Layout(width="300px"))
monitor_output = widgets.Output()

def refresh_training_monitor(_=None):
    with monitor_output:
        clear_output(wait=True)
        state = monitor_read_json(MONITOR_QUEUE / "queue_status.json")
        extension_state = monitor_read_json(MONITOR_QUEUE / "mask90_scaling_status.json")
        if extension_state.get("updated_unix", 0) > state.get("updated_unix", 0):
            state = extension_state
        plan = monitor_read_json(MONITOR_QUEUE / "plan.json")
        stage_names = {"preparing": "Preparazione manifest", "preflight": "Verifica CUDA/bf16",
            "training": "Training", "preparing40000": "Preparazione 40.000 crop",
            "training40000": "Training 40.000 crop", "completed": "Coda completata", "failed": "Coda fermata per errore"}
        stage = state.get("stage", "Stato non ancora disponibile")
        active_run = "large40000" if stage == "training40000" else state.get("run")
        print(f"Aggiornato: {datetime.now().astimezone():%d/%m/%Y %H:%M:%S %Z}")
        print(f"Coda: {stage_names.get(stage, stage)}; run: {active_run or '—'}")
        if state.get("error"):
            print(f"Errore: {state['error']}")
        policy = monitor_read_json(MONITOR_QUEUE / "expansion_policy.json")
        if policy.get("mode") == "prepare_only":
            print("Training aggiuntivo sul nuovo dataset disabilitato.")
        if active_run:
            print("Ultima riga del log:", monitor_log_tail(MONITOR_QUEUE / active_run / "training.log"))
        try:
            evaluations = pd.read_csv(MONITOR_QUEUE / "validation_mask_sweep.csv")
        except (FileNotFoundError, pd.errors.EmptyDataError, pd.errors.ParserError):
            evaluations = pd.DataFrame()
        figure, axis = plt.subplots(figsize=(11, 5))
        rows = []
        sparse_labels = []
        for name, label, color in MONITOR_RUNS:
            directory = MONITOR_QUEUE / name
            frame = monitor_read_epochs(directory / "loss_epochs.csv")
            progress = monitor_read_json(directory / "progress.json")
            config = monitor_read_json(directory / "run_config.json").get("arguments", {})
            training_mask = config.get("mask_ratio", {"mask50": .5, "mask75": .75, "mask90": .9}.get(name))
            requested = plan.get("large_epochs" if name == "large40000" else "epochs", 150)
            completed = int(frame.epoch.max()) if not frame.empty else 0
            status = "In corso" if active_run == name else "Non iniziato"
            if progress.get("status") == "completed":
                status = "Completato"
            elif completed and active_run != name:
                status = "Interrotto / in attesa"
            if stage == "failed" and active_run == name:
                status = "Errore"
            if name == "large40000" and policy.get("mode") == "prepare_only":
                status = "Disabilitato: solo preparazione dataset"
            rows.append({"Run": label, "Stato": status, "Epoche": f"{completed}/{requested}"})
            if frame.empty or training_mask is None:
                continue
            ratio = training_mask if monitor_validation_mode.value == "own" else monitor_validation_mode.value
            run_identity = monitor_read_json(directory / "run_config.json").get("identity", {})
            loss_scale = float(run_identity.get("std", 1.)) ** 2
            axis.plot(frame.epoch, frame.mean_loss * loss_scale, color=color, label=f"Training · {label}")
            history = pd.DataFrame(columns=["epoch", "loss"])
            if ratio == .9 and "validation_mse" in frame:
                history = frame[["epoch", "validation_mse"]].rename(columns={"validation_mse": "loss"}).dropna()
            elif ratio == training_mask and "validation_own_mse" in frame:
                history = frame[["epoch", "validation_own_mse"]].rename(columns={"validation_own_mse": "loss"}).dropna()
            if not history.empty:
                axis.plot(history.epoch, history.loss * loss_scale, "--", color=color,
                    label=f"Validation · modello {training_mask:.0%}, valutato {ratio:.0%}")
            points = pd.DataFrame()
            if not evaluations.empty:
                points = evaluations[(evaluations.run == name) &
                    np.isclose(evaluations.validation_mask_ratio, ratio)].copy()
                if not points.empty:
                    points = points.sort_values(["epoch", "checkpoint"]).drop_duplicates("epoch", keep="last")
                    # Only display checkpoint points that do not already have an epoch value.
                    points = points[~points.epoch.isin(history.epoch)]
            if not points.empty:
                axis.plot(points.epoch, points.validation_mse * loss_scale, "o--", color=color,
                    label=f"Validation · modello {training_mask:.0%}, valutato {ratio:.0%} (checkpoint)")
                sparse_labels.append(f"modello {training_mask:.0%}")
            elif history.empty:
                print(f"Validation {ratio:.0%} per modello {training_mask:.0%}: valutazione non ancora disponibile.")
        display(pd.DataFrame(rows).set_index("Run")[["Stato", "Epoche"]])
        if sparse_labels:
            print("Validation da soli checkpoint disponibili per " + ", ".join(sparse_labels) +
                  ": i cerchi sono misure reali; il tratteggio li collega, senza ricostruire le epoche intermedie.")
        mode_label = "proprio masking" if monitor_validation_mode.value == "own" else f"{monitor_validation_mode.value:.0%} per tutti"
        axis.set(title=f"VideoMAE V2 Small — encoder: {mode_label}; decoder: 50%", xlabel="Epoca", ylabel="MSE (K²)")
        axis.grid(alpha=.3)
        if axis.lines:
            axis.legend(fontsize=8)
        figure.tight_layout(); display(figure); plt.close(figure)
        comparison_path = MONITOR_QUEUE / "mask90_scaling_comparison.csv"
        if not comparison_path.exists():
            comparison_path = MONITOR_QUEUE / "dataset_comparison.csv"
        if comparison_path.exists():
            comparison = pd.read_csv(comparison_path)
            fig, ax = plt.subplots(figsize=(7, 4))
            for ratio, group in comparison.groupby("training_mask"):
                ax.plot(group.samples, group.rmse_k ** 2, "o-", label=f"Encoder training {ratio:.0%}")
            ax.legend()
            ax.set(xlabel="Clip di training", ylabel="Validation MSE (K²)",
                   title="Effetto della dimensione del dataset — encoder90%, decoder50%")
            ax.set_xticks(comparison.samples); ax.grid(alpha=.3)
            display(fig); plt.close(fig)

async def training_monitor_loop():
    try:
        while monitor_auto.value:
            await asyncio.sleep(30)
            if monitor_auto.value:
                refresh_training_monitor()
    except asyncio.CancelledError:
        pass

def toggle_training_monitor(change):
    global training_monitor_task
    if change["new"]:
        refresh_training_monitor()
        training_monitor_task = asyncio.get_running_loop().create_task(training_monitor_loop())
    elif training_monitor_task is not None:
        training_monitor_task.cancel(); training_monitor_task = None

monitor_refresh_button.on_click(refresh_training_monitor)
monitor_auto.observe(toggle_training_monitor, names="value")
monitor_validation_mode.observe(refresh_training_monitor, names="value")
display(widgets.HBox([monitor_refresh_button, monitor_auto, monitor_validation_mode]), monitor_output)
refresh_training_monitor()


## Ricostruzione VideoMAE sulle clip di training

Eseguire le due celle seguenti con il kernel `.venv`. Nel player scegliere **Modello** (training 50%, 75%, 90% o il precedente run), **Checkpoint** (migliore sulla validation al 90% oppure finale) e **Mask** (proprio masking o 50/75/90%). Default: modello al 75%, checkpoint migliore, proprio masking. Il cambio modello libera il precedente dalla GPU e svuota la cache delle ricostruzioni.

**Play** scorre i 16 frame con una tube mask fissata per clip, condivisa fra modelli alla stessa percentuale. I tre nuovi modelli usano lo stesso manifest: scegliere la stessa clip e la stessa percentuale per confrontarli. Il precedente run usa invece il suo manifest originale. Media/deviazione standard e manifest vengono sempre letti dal rispettivo run e verificati contro il checkpoint.

I pixel visibili sono copiati dall'originale; errori e metriche in Kelvin riguardano solo quelli mascherati. Bande: scala fissa 180–330 K; errori: ±20 K. Queste sono clip di training e servono alla diagnostica, non alla misura della generalizzazione. Il checkpoint «migliore» è selezionato per validation al 90%, non per una valutazione retrospettiva con altre percentuali.


I nuovi run sono marcati **V2 Small**, quelli precedenti **V1 Small**. Per visualizzare una clip completa, il player V2 mantiene il masking encoder scelto e disattiva soltanto il sottocampionamento del decoder: ricostruisce tutte le patch nascoste, con gli stessi pesi. La diagnostica completa può quindi differire dalla loss di training/validation V2, misurata su un sottoinsieme running cell. I run futuri diventano disponibili dopo il primo checkpoint.

In [ ]:
from pathlib import Path
from functools import lru_cache
import hashlib
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
from transformers import VideoMAEConfig, VideoMAEForPreTraining

RECON_ROOT = Path.cwd()
if not (RECON_ROOT / 'src').exists() and (RECON_ROOT.parent / 'src').exists():
    RECON_ROOT = RECON_ROOT.parent
sys.path.insert(0, str(RECON_ROOT / 'src'))
from emma_gpm.seviri import THERMAL_CHANNELS
from emma_gpm.seviri_clips import SeviriZarrClipReader, normalize_seviri_clip
from emma_gpm.videomae import forward_seviri_mae, restore_masked_videomae_pixels
from emma_gpm.pretraining_validation import fixed_tube_mask
from emma_gpm.videomae_v2 import VideoMAEV2ForPreTraining

RECON_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RECON_MASK_SEED = 20261001
RECON_MAX_EXAMPLES = 100
RECON_RUNS = {
    'mask50': ('V1 Small · training mask 50%', RECON_ROOT / 'results/videomae_pretraining_smoke_masking_sweep/mask50'),
    'mask75': ('V1 Small · training mask 75%', RECON_ROOT / 'results/videomae_pretraining_smoke_masking_sweep/mask75'),
    'mask90': ('V1 Small · training mask 90%', RECON_ROOT / 'results/videomae_pretraining_smoke_masking_sweep/mask90'),
    'legacy': ('V1 Small · precedente run 150 epoche', RECON_ROOT / 'results/videomae_pretraining_smoke_large4096/train_onfly_150'),
}
# Aggiungi i nuovi run V2 senza nascondere i precedenti V1.
for name in ['mask50', 'mask75', 'mask90', 'large4000', 'large8000', 'large4000_mask90', 'large8000_mask90']:
    path = RECON_ROOT / 'results/videomae_v2_small_scaling' / name
    RECON_RUNS['v2_' + name] = ('V2 Small · ' + name, path)
recon_switching = False

def load_reconstruction_model(key, checkpoint_kind):
    global recon_model, recon_reader, recon_manifest, recon_examples, recon_stats
    global recon_epoch, recon_mean, recon_std, recon_training_mask, RECON_RUN
    old_reader = globals().get('recon_reader')
    if old_reader is not None:
        old_reader.close()
    cache = globals().get('reconstruct_training_clip')
    if cache is not None:
        cache.cache_clear()
    if globals().get('recon_model') is not None:
        recon_model = None
        if RECON_DEVICE == 'cuda':
            torch.cuda.empty_cache()
    _, RECON_RUN = RECON_RUNS[key]
    run_config = json.loads((RECON_RUN / 'run_config.json').read_text())
    arguments = run_config['arguments']
    recon_manifest_path = Path(arguments['manifest'])
    checkpoint_path = RECON_RUN / ('checkpoint_best.pt' if checkpoint_kind == 'best' else 'checkpoint_latest.pt')
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        identity = checkpoint['identity']
        if hashlib.sha256(recon_manifest_path.read_bytes()).hexdigest() != identity['manifest_sha256']:
            raise ValueError('Il manifest è diverso da quello del checkpoint.')
        model_class = VideoMAEV2ForPreTraining if identity['config'].get('videomae_version') == 'v2' else VideoMAEForPreTraining
        recon_model = model_class(VideoMAEConfig(**identity['config']))
        recon_model.load_state_dict(checkpoint['model'])
        recon_stats = dict(global_mean_k=identity['mean'], global_std_k=identity['std'],
                           zarr_root=identity['zarr_root'], channels=list(THERMAL_CHANNELS))
        recon_epoch = checkpoint['progress']['completed_epochs']
        recon_training_mask = identity['mask_ratio']
        del checkpoint
    else:
        if checkpoint_kind == 'best':
            raise FileNotFoundError('Checkpoint migliore non disponibile per questo run.')
        model_config = VideoMAEConfig.from_pretrained(RECON_RUN / 'model')
        model_class = VideoMAEV2ForPreTraining if getattr(model_config, 'videomae_version', 'v1') == 'v2' else VideoMAEForPreTraining
        recon_model = model_class.from_pretrained(RECON_RUN / 'model')
        recon_stats = json.loads((RECON_RUN / 'channel_stats.json').read_text())
        recon_epoch = json.loads((RECON_RUN / 'smoke_summary.json').read_text())['epochs']
        recon_training_mask = arguments['mask_ratio']
    assert recon_model.config.num_channels == 7 and not recon_model.config.norm_pix_loss
    assert tuple(recon_stats['channels']) == THERMAL_CHANNELS
    recon_mean, recon_std = recon_stats['global_mean_k'], recon_stats['global_std_k']
    zarr_root = Path(os.environ.get('SEVIRI_ZARR_ROOT', recon_stats['zarr_root']))
    recon_manifest = pd.read_csv(recon_manifest_path)
    recon_examples = recon_manifest.head(RECON_MAX_EXAMPLES)
    recon_reader = SeviriZarrClipReader(zarr_root)
    recon_model = recon_model.to(RECON_DEVICE).eval()


In [ ]:
@lru_cache(maxsize=2)
def reconstruct_training_clip(index):
    row = recon_manifest.iloc[index]
    original = recon_reader.load(row.start_time, origin_y=int(row.origin_y), origin_x=int(row.origin_x))
    normalized = normalize_seviri_clip(original, recon_mean, recon_std)
    pixels = torch.from_numpy(np.ascontiguousarray(normalized)).unsqueeze(0).to(RECON_DEVICE)
    config = recon_model.config
    ratio = recon_training_mask if recon_mask_selector.value == 'own' else recon_mask_selector.value
    mask = fixed_tube_mask(config, [index], ratio, RECON_MASK_SEED, RECON_DEVICE)
    with torch.inference_mode():
        predicted = forward_seviri_mae(recon_model, pixels, mask)
        restored, pixel_mask = restore_masked_videomae_pixels(pixels, predicted.logits, mask, config)
    reconstructed = restored[0].cpu().numpy() * recon_std + recon_mean
    hidden = pixel_mask[0, :, 0].cpu().numpy()
    # Copia esplicita in Kelvin evita anche il roundoff sui pixel non predetti.
    reconstructed = np.where(hidden[:, None], reconstructed, original)
    metrics = []
    for channel, label in enumerate(THERMAL_CHANNELS):
        error = (reconstructed[:, channel] - original[:, channel])[hidden]
        baseline_error = (recon_mean - original[:, channel])[hidden]
        metrics.append(dict(channel=label, masked_mae_k=float(np.abs(error).mean()),
                            masked_rmse_k=float(np.sqrt(np.square(error).mean())),
                            global_mean_baseline_rmse_k=float(np.sqrt(np.square(baseline_error).mean()))))
    return original, reconstructed, hidden, pd.DataFrame(metrics)

recon_model_selector = widgets.Dropdown(options=[(label, key) for key, (label, _) in RECON_RUNS.items()],
    value=('v2_mask50' if (RECON_RUNS['v2_mask50'][1] / 'checkpoint_latest.pt').exists() else 'mask75'), description='Modello:', layout=widgets.Layout(width='370px'))
recon_checkpoint_selector = widgets.Dropdown(options=[('Migliore (validation 90%)', 'best'), ('Finale', 'final')],
    value='best', description='Checkpoint:', layout=widgets.Layout(width='330px'))
recon_mask_selector = widgets.Dropdown(options=[('Proprio masking', 'own'), ('50%', .5), ('75%', .75), ('90%', .9)],
    value='own', description='Mask:', layout=widgets.Layout(width='250px'))
recon_selector = widgets.Dropdown(options=[], description='Clip:', layout=widgets.Layout(width='320px'))
recon_play = widgets.Play(value=0, min=0, max=15, step=1, interval=1000, description='Play')
recon_slider = widgets.IntSlider(value=0, min=0, max=15, description='Frame:', continuous_update=False,
                                layout=widgets.Layout(width='520px'))
widgets.jslink((recon_play, 'value'), (recon_slider, 'value'))
recon_output = widgets.Output()
recon_metrics_output = widgets.Output()
recon_bt_cmap = plt.get_cmap('inferno').copy()
recon_bt_cmap.set_bad('#888888')
recon_error_cmap = plt.get_cmap('RdBu_r').copy()
recon_error_cmap.set_bad('#888888')

def show_reconstruction_frame(*_):
    if recon_switching or recon_selector.value is None:
        return
    index, frame = recon_selector.value, recon_slider.value
    original, restored, hidden, _ = reconstruct_training_clip(index)
    row = recon_manifest.iloc[index]
    timestamp = pd.Timestamp(row.start_time) + pd.Timedelta(minutes=5 * frame)
    with recon_output:
        clear_output(wait=True)
        figure, axes = plt.subplots(4, 7, figsize=(23, 12), constrained_layout=True)
        row_labels = ['Originale (K)', 'Input: mascherati grigi', 'Ricostruita + visibili copiati (K)', 'Errore solo mascherati (K)']
        for channel, label in enumerate(THERMAL_CHANNELS):
            images = [original[frame, channel], np.where(hidden[frame], np.nan, original[frame, channel]),
                      restored[frame, channel], np.where(hidden[frame], restored[frame, channel]-original[frame, channel], np.nan)]
            for r, values in enumerate(images):
                axis = axes[r, channel]
                image = axis.imshow(values, origin='upper', cmap=recon_error_cmap if r == 3 else recon_bt_cmap,
                                    vmin=-20 if r == 3 else 180, vmax=20 if r == 3 else 330)
                axis.set_xticks([]); axis.set_yticks([])
                if r == 0: axis.set_title(label)
                if channel == 0: axis.set_ylabel(row_labels[r])
                if channel == 6: figure.colorbar(image, ax=axes[r, :].tolist(), shrink=.8, label='K')
        figure.suptitle(f"{row.clip_id} | {timestamp:%Y-%m-%d %H:%M UTC} | {recon_model_selector.label} | epoca {recon_epoch} | tube mask {hidden.mean():.1%}")
        display(figure)
        plt.close(figure)

def select_reconstruction_clip(*_):
    if recon_switching or recon_selector.value is None:
        return
    recon_play.value = 0
    _, _, _, metrics = reconstruct_training_clip(recon_selector.value)
    with recon_metrics_output:
        clear_output(wait=True)
        print('Errori sui pixel mascherati di tutti i 16 frame; baseline = media globale del training.')
        display(metrics.round(3))
    show_reconstruction_frame()

def switch_reconstruction_model(*_):
    global recon_switching
    if recon_switching:
        return
    recon_switching = True
    try:
        recon_play.value = 0
        key = recon_model_selector.value
        run = RECON_RUNS[key][1]
        choices = [('Migliore (validation 90%)', 'best'), ('Finale', 'final')] if (run / 'checkpoint_best.pt').exists() else [('Finale', 'final')]
        if not (run / 'checkpoint_latest.pt').exists() and not (run / 'model').exists():
            recon_selector.options = []
            with recon_metrics_output:
                clear_output(wait=True)
            with recon_output:
                clear_output(wait=True)
                print('Questo run non ha ancora un checkpoint. Attendere la prima epoca e riselezionarlo.')
            return
        requested_kind = recon_checkpoint_selector.value
        recon_checkpoint_selector.options = choices
        recon_checkpoint_selector.value = requested_kind if requested_kind in [value for _, value in choices] else 'final'
        load_reconstruction_model(key, recon_checkpoint_selector.value)
        recon_selector.options = [(row.clip_id, int(i)) for i, row in recon_examples.iterrows()]
        recon_selector.value = int(recon_examples.index[0])
    finally:
        recon_switching = False
    select_reconstruction_clip()

def switch_reconstruction_mask(*_):
    reconstruct_training_clip.cache_clear()
    select_reconstruction_clip()

recon_model_selector.observe(switch_reconstruction_model, names='value')
recon_checkpoint_selector.observe(switch_reconstruction_model, names='value')
recon_mask_selector.observe(switch_reconstruction_mask, names='value')
recon_selector.observe(select_reconstruction_clip, names='value')
recon_slider.observe(show_reconstruction_frame, names='value')
display(widgets.HBox([recon_model_selector, recon_checkpoint_selector, recon_mask_selector]),
        widgets.HBox([recon_selector, recon_play, recon_slider]), recon_metrics_output, recon_output)
switch_reconstruction_model()
